In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import glob
import pdfplumber
from datetime import datetime
import re
import numpy as np

In [3]:
def extract(filepath):
    """
    Extracts structured data from an Excel file with three sheets:
    'Summary', 'Payout Breakup', and 'Order Level'.

    Parameters:
        filepath (str): Path to the Excel file.

    Returns:
        tuple:
            - df_summary (DataFrame): Metadata extracted from the 'Summary' sheet.
            - df_payout (DataFrame): Cleaned payout details from 'Payout Breakup' sheet.
            - df_order (DataFrame): Cleaned order-level data from 'Order Level' sheet.
    """
    df_summary = pd.read_excel(filepath, sheet_name="Summary", header=None, skiprows=4)
    df_payout = pd.read_excel(filepath, sheet_name="Payout Breakup", header=None, skiprows=1)
    df_order = pd.read_excel(filepath, sheet_name="Order Level", header=None, skiprows=2)

    # Extracting the summary information
    summary_dict = {
        "Brand": df_summary.iloc[0, 1],
        "Location": df_summary.iloc[1, 1],
        "City": df_summary.iloc[2, 1],
        "Res-id": df_summary.iloc[3, 1].split("- ")[1],
        df_summary.iloc[7, 1]: df_summary.iloc[7, 2],
        df_summary.iloc[8, 1]: df_summary.iloc[8, 2],
        df_summary.iloc[9, 1]: df_summary.iloc[9, 2].strip(),
        df_summary.iloc[10, 1]: df_summary.iloc[10, 2].strip(),
        df_summary.iloc[11, 1]: df_summary.iloc[11, 2].strip(),
        "File Name": filepath.split("/")[-1]
    }
    df_summary =  pd.DataFrame(summary_dict, index=[0])

    # Extracting the payout information
    df_payout = df_payout.drop([2, 3, 9, 21, 25, 27, 32]).drop(columns=[0, 1]).dropna(how='all').reset_index(drop=True)
    df_payout.columns = df_payout.iloc[0]
    df_payout = df_payout.drop(0)
    df_payout['Brand'] = df_summary['Brand'][0]
    df_payout['Res-id'] = df_summary['Res-id'][0]
    df_payout['Payout Period'] = df_summary['Payout Period'][0]
    df_payout['File Name'] = df_summary['File Name'][0]
    df_payout['SR.No'] = np.arange(1, len(df_payout) + 1)
    columns = ['SR.No'] + [col for col in df_payout.columns if col not in ['SR.No']]
    df_payout = df_payout[columns]
    
    # Extracting the order level information
    df_order.columns = df_order.iloc[0]
    df_order = df_order.drop(0)
    df_order['Brand'] = df_summary['Brand'][0]
    df_order['Res-id'] = df_summary['Res-id'][0]
    df_order['Payout Period'] = df_summary['Payout Period'][0]
    df_order['File Name'] = df_summary['File Name'][0]
    cols = ['Brand', 'Res-id', 'Payout Period', 'File Name'] + [col for col in df_order.columns if col not in ['Brand', 'Res-id', 'Payout Period', 'File Name']]
    df_order = df_order[cols]

    return df_summary, df_payout, df_order



In [4]:
def extract_pdf_data(pdf_path):
    """
    Extracts structured invoice data from a PDF file.

    Parameters:
        pdf_path (str): Path to the invoice PDF.

    Returns:
        DataFrame: Cleaned invoice details including metadata and line items,
                   with derived fields like fiscal year, month, and payout period.
    """
    with pdfplumber.open(pdf_path) as pdf:
        first_page = pdf.pages[0]

        # Extract table
        table = first_page.extract_table()
        df = pd.DataFrame(table[4:], columns=table[3])
        
        if df.iloc[1, 0] == '2':
            custom_data  = {
            df.iloc[4,0]: df.iloc[4,4],
            df.iloc[5,0]: df.iloc[5,4],
            }
        else:
            custom_data  = {
            df.iloc[3,0]: df.iloc[3,4],
            df.iloc[4,0]: df.iloc[4,4],
            }
        for key, value in custom_data.items():
            df[key] = value

        # Keep only relevant rows
        df = df.iloc[:2, :]

        # Clean column names
        df.columns = [col.lower().replace('\n', '_').replace(' ', '_').replace('-', '').replace('__', '_').replace('.', '').replace('_(rs)', '') for col in df.columns]

        # Clean description text
        df['description'] = df['description'].str.replace(r'\n', ' ', regex=True)

        # Extract text from the page
        text = first_page.extract_text()

    # Define regex patterns to extract the needed fields
    patterns = {
        'brand_id': r'Restaurant / Store ID\s*:\s*(.*?)\s*GSTIN',  
        'pan': r'PAN\s*:\s*(\S+)',
        'invoice_date': r'Invoice Date\s*:\s*(.*?)\s*GSTIN',  
        'invoice_number': r'Invoice Number\s*:\s*(\S+)',
        'original_invoice_number': r'Original Invoice\s*-\s*No\s*:\s*(.*?)\s*No:',  
        'invoice_type': r'Invoice Type\s*:\s*(\S+)',
        'payout_period': r'Service Period\s*:\s*(.*?)\s*Address', 
        'irn': r'IRN\s*:\s*(\w+)',
        'mann_gstin': r'Invoice To\s*:\s*.*?GSTIN\s*:\s*(29[A-Z0-9]+)',  
        'swiggy_gstin': r'GSTIN\s*:\s*(29[A-Z0-9]+)',  
    }

    # Extract data into a dictionary
    invoice_data = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.DOTALL)
        invoice_data[key] = match.group(1).strip() if match else 'nan'

    # Parse and reformat dates
    invoice_dt = datetime.strptime(invoice_data['invoice_date'], "%Y-%m-%d %H:%M:%S")
    invoice_data['invoice_date'] = invoice_dt.strftime("%-d/%-m/%Y %-I:%M:%S %p")

    # Format payout_period
    start_str, end_str = invoice_data['payout_period'].split(' to ')
    start_dt = datetime.strptime(start_str.strip(), "%d/%m/%Y")
    end_dt = datetime.strptime(end_str.strip(), "%d/%m/%Y")
    invoice_data['payout_period'] = f"{start_dt.day} {start_dt.strftime('%B')} {start_dt.year} - {end_dt.day} {end_dt.strftime('%B')} {end_dt.year}"

    # Add more derived fields
    invoice_data['year'] = invoice_dt.year
    invoice_data['month'] = f"{invoice_dt.month}_{invoice_dt.strftime('%B')}"
    fy_start_year = invoice_dt.year if invoice_dt.month >= 4 else invoice_dt.year - 1
    fy_end_year = fy_start_year + 1
    invoice_data['fy_year'] = f"FY_{fy_start_year}-{fy_end_year}"
    invoice_data['file_name'] = pdf_path.split("/")[-1]

    # Add invoice_data fields to each row in df
    for key, value in invoice_data.items():
        df[key] = value
    df.reset_index(drop=True, inplace=True)
    # Reorder columns
    columns = ['payout_period', 'file_name', 'fy_year', 'year', 'month', 'irn',
        'mann_gstin', 'swiggy_gstin', 'sr_no', 'description', 'hsn',
        'unit_of_measure', 'quantity', 'unit_price', 'base_amount', 'discount',
        'assessable_value', 'cgst_rate', 'cgst_amount', 'sgst_rate',
        'sgst_amount', 'igst_rate', 'igst_amount', 'comp_cess_rate',
        'comp_cess_amount', 'state_cess_rate', 'state_cess_amount',
        'total_amount', 'other_charges_reimbursement_of_discount',
        'grand_total', 'brand_id', 'pan', 'invoice_date', 'invoice_number',
        'original_invoice_number', 'invoice_type']
    df = df[columns]
    return df

In [5]:
def extract_swiggy_data(excel_path):
    """
    Loads and filters Swiggy invoice data from an Excel file.

    Parameters:
        excel_path (str): Path to the Swiggy Excel file.

    Returns:
        DataFrame: Filtered invoice data with standardized columns.
    """
    swiggy_df = pd.read_excel(excel_path, header=0)
    columns = ['payout_period', 'file_name', 'fy_year', 'year', 'month', 'irn',
            'mann_gstin', 'swiggy_gstin', 'sr_no', 'description', 'hsn',
            'unit_of_measure', 'quantity', 'unit_price', 'base_amount', 'discount',
            'assessable_value', 'cgst_rate', 'cgst_amount', 'sgst_rate',
            'sgst_amount', 'igst_rate', 'igst_amount', 'comp_cess_rate',
            'comp_cess_amount', 'state_cess_rate', 'state_cess_amount',
            'total_amount', 'other_charges_reimbursement_of_discount',
            'grand_total', 'brand_id', 'pan', 'invoice_date', 'invoice_number',
            'original_invoice_number', 'invoice_type']
    swiggy_df = swiggy_df[columns]
    return swiggy_df

In [6]:
def compile_excel_report():
    """
    Compiles data from multiple sources into a single Excel report with 4 sheets.

    Sources:
        - Excel summary files from 'Payout Summary & Order Level Sales/'
        - PDF commission invoices from 'Commission Invoices/'
        - Swiggy commission data from a specified Excel file

    Output:
        - Creates 'Final_Compiled_Report.xlsx' containing the following sheets:
            1. Summary
            2. Payout Breakup Tab
            3. Order Level
            4. Commission Invoice

    Each sheet aggregates data from its respective source(s) into a cleaned, standardized format.
    """
    summary_folder_path = "Payout Summary & Order Level Sales/"
    commission_folder_path = "Commission Invoices/"
    swiggy_excel_path = "Commission Invoices/Swiggy_Tax_Sample_file.xlsx"
    output_file = "Final_Compiled_Report.xlsx"

    # Get all Excel and PDF files
    excel_files = glob.glob(summary_folder_path + "*.xlsx")
    pdf_files = glob.glob(commission_folder_path + "*.pdf")

    # Initialize containers
    summary_list = []
    payout_list = []
    order_list = []
    pdf_list = []

    # Process Excel summary files
    for filepath in excel_files:
        summary_row, payout_rows, order_row = extract(filepath)
        summary_list.append(summary_row)
        payout_list.append(payout_rows)
        order_list.append(order_row)

    # Process PDF files
    for filepath in pdf_files:
        pdf_row = extract_pdf_data(filepath)
        pdf_list.append(pdf_row)

    # Extract Swiggy data
    swiggy_df = extract_swiggy_data(swiggy_excel_path)

    # Combine all data into final DataFrames
    final_summary_df = pd.concat(summary_list, ignore_index=True)
    final_payout_df = pd.concat(payout_list, ignore_index=True)
    final_order_df = pd.concat(order_list, ignore_index=True)
    final_commission_df = pd.concat(pdf_list + [swiggy_df], ignore_index=True)

    # Write to Excel with 4 sheets
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        final_summary_df.to_excel(writer, sheet_name="Summary", index=False)
        final_payout_df.to_excel(writer, sheet_name="Payout Breakup Tab", index=False)
        final_order_df.to_excel(writer, sheet_name="Order Level", index=False)
        final_commission_df.to_excel(writer, sheet_name="Commission Invoice", index=False)

    print(f"Excel file '{output_file}' created with 4 sheets.")



In [ ]:
compile_excel_report()

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

Excel file 'Final_Compiled_Report.xlsx' created with 4 sheets.
